In [ ]:
!uv pip install pydot -q

# Autoencoders for Reconstructions

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the **encoder-decoder architecture** of autoencoders
- Recognize how the **bottleneck layer** forces compression of information
- Train autoencoders for **image reconstruction**
- Apply **denoising autoencoders** to clean corrupted data
- Understand the key difference between standard Autoencoders and **Variational Autoencoders (VAEs)**
- Observe how VAEs can **generate new samples** from learned distributions

An autoencoder is a special type of neural network that is trained to copy its input to its output. For example, given an image of a handwritten digit, an autoencoder first encodes the image into a lower dimensional latent representation, then decodes the latent representation back to an image. An autoencoder learns to compress the data while minimizing the reconstruction error.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import mnist
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 1. Load and Preprocess Data
(x_train, _), (x_test, _) = mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

## Define Model Architecture
## Encoder

### What to Expect: Building the Encoder

**The Encoder's Role:** Think of the encoder as a *compression assistant*. It takes a full 28x28 image (784 pixels) and progressively reduces it to a much smaller representation (32 values in our case).

**Watch For:**
- **Input Layer**: Takes the 28×28 MNIST image
- **Flatten Layer**: Converts the 2D image to a 1D vector of 784 values
- **Dense Layers**: Progressively compress: 784 → 128 → 64 → 32
- **Bottleneck**: The final 32-dimensional representation contains the "essence" of the image

**Key Insight:** The bottleneck forces the model to learn the most important features needed to reconstruct the image.

In [ ]:
latent_dim = 32  # Size of the compressed representation

# Encoder
input_img = Input(shape=(28, 28), name='encoder_input')
flattened = Flatten(name='encoder_flatten')(input_img)
encoded1 = Dense(128, activation='relu', name='encoder_dense1')(flattened)
encoded2 = Dense(64, activation='relu', name='encoder_dense2')(encoded1)
bottleneck = Dense(latent_dim, activation='relu', name='bottleneck')(encoded2)

encoder = Model(input_img, bottleneck, name='encoder')
encoder.summary()

## Decoder

### What to Expect: Building the Decoder

**The Decoder's Role:** The decoder does the *opposite* of the encoder—it takes the compressed representation and reconstructs the original image.

**Watch For:**
- The layer sizes are **reversed**: 32 → 64 → 128 → 784
- **Sigmoid activation** on the output ensures values are between 0 and 1 (matching our normalized pixel values)
- **Reshape** converts the 784-dimensional output back to 28×28 image format

**Key Insight:** The decoder learns to "decompress" the bottleneck representation. If the encoder learned good features, the decoder can produce high-quality reconstructions.

In [ ]:
decoder_input = Input(shape=(latent_dim,), name='decoder_input')
decoded1 = Dense(64, activation='relu', name='decoder_dense1')(decoder_input)
decoded2 = Dense(128, activation='relu', name='decoder_dense2')(decoded1)
decoded3 = Dense(784, activation='sigmoid', name='decoder_output')(decoded2)
reconstructed = Reshape((28, 28), name='decoder_reshape')(decoded3)

decoder = Model(decoder_input, reconstructed, name='decoder')
decoder.summary()

## Combining encoder and decoder

In [ ]:
# Autoencoder (combining encoder and decoder)
autoencoder_input = Input(shape=(28, 28), name='autoencoder_input')
encoded_repr = encoder(autoencoder_input)
reconstructed_img = decoder(encoded_repr)
autoencoder = Model(autoencoder_input, reconstructed_img, name='autoencoder')
autoencoder.summary()

In [ ]:
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(autoencoder, to_file='model_plot.png', show_shapes=True, show_layer_names=True)



## Train the model

In [ ]:
# 3. Train the Autoencoder
autoencoder.fit(x_train, x_train,
                epochs=10,
                batch_size=256,
                shuffle=True,
                validation_data=(x_test, x_test))


In [ ]:
# 4. Visualize the Reconstructions
decoded_imgs = autoencoder.predict(x_test)

In [ ]:
n = 10  # Number of digits to display
plt.figure(figsize=(20, 4))
for i in range(n):
    # Display original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # Display reconstruction
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
plt.show()

# Denoising Autoencoders
A denoising autoencoder is trained to remove noise from its input. Instead of reconstructing the exact input, it learns to map noisy data to clean data. This is useful for tasks like image denoising, where the goal is to recover the original image from a corrupted version.

### What to Expect: Denoising Autoencoder

**The Goal:** Train a model to *remove noise* from corrupted images.

**How It Works:**
1. We **add random noise** to the input images
2. The autoencoder learns to map noisy images → clean images
3. This forces the model to learn **robust features** that ignore noise

**What You'll See:**
- **Original**: Clean MNIST digits
- **Noisy**: Same digits with added Gaussian noise
- **Denoised**: The autoencoder's cleaned output

**Key Insight:** Before training, the denoised images look random. After training, they should closely match the originals—proving the model learned to filter out noise.

In [ ]:
# 1. Add noise to the MNIST images
noise_factor = 0.5
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)
x_train_noisy = np.clip(x_train_noisy, 0., 1.)
x_test_noisy = np.clip(x_test_noisy, 0., 1.)

We show three rows for each sample:

* __Original__: The clean MNIST digit.
* __Noisy__: The same digit with added noise.
* __Denoised__: The output from the autoencoder when given the noisy image.

If you run this cell before training, the "denoised" images are just the autoencoder's random output—they don't look clean yet. This helps you see the baseline: the model can't denoise before learning.

After training, the "denoised" images should look much closer to the originals, showing the autoencoder has learned to remove noise.



In [ ]:
# Visualize denoising before training (weights are random)
sample_denoised_imgs = autoencoder.predict(x_test_noisy)
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    # Original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Original', fontsize=14)
    # Noisy
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Noisy', fontsize=14)
    # Denoised (before training)
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(sample_denoised_imgs[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Denoised (init)', fontsize=14)
plt.show()

In [ ]:
# 2. Train the autoencoder for denoising
autoencoder.fit(x_train_noisy, x_train,
                epochs=10,
                batch_size=256,
                shuffle=True,
                validation_data=(x_test_noisy, x_test))

### 3. Evaluate Denoising Performance

Now that the model is trained, let's look at its performance on the test set. We will display:
1.  **Original Images**: The clean MNIST digits.
2.  **Noisy Images**: The input to the autoencoder (with added Gaussian noise).
3.  **Denoised Images**: The reconstruction output from the autoencoder.

Compare the Denoised images with the Original ones to see how effective the model is at removing noise.

In [ ]:
# 3. Visualize original, noisy, and denoised images
denoised_imgs = autoencoder.predict(x_test_noisy)
n = 10
plt.figure(figsize=(20, 6))
for i in range(n):
    # Original
    ax = plt.subplot(3, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Original', fontsize=14)
    # Noisy
    ax = plt.subplot(3, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Noisy', fontsize=14)
    # Denoised
    ax = plt.subplot(3, n, i + 1 + 2*n)
    plt.imshow(denoised_imgs[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Denoised', fontsize=14)
plt.show()

# Student Activity: Autoencoder with Fashion MNIST
In this activity, you will apply the same autoencoder architecture to the Fashion MNIST dataset. Your goal is to reconstruct Fashion MNIST images using an autoencoder, just as we did with MNIST digits.

**Steps:**
1. Load and preprocess the Fashion MNIST dataset.
2. Define the encoder and decoder models (you can reuse the architecture from above).
3. Train the autoencoder to reconstruct Fashion MNIST images.
4. Visualize the original and reconstructed Fashion MNIST images.

Try to experiment with different latent dimensions or architectures to see how reconstruction quality changes!

In [ ]:
fashion_mnist = tf.keras.datasets.fashion_mnist

(x_train, _), (x_test, _) = fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

print (x_train.shape)
print (x_test.shape)

In [ ]:
# 2. Define Model Architecture
latent_dim = 32  # Size of the compressed representation

# Encoder
input_img = Input(shape=(28, 28), name='encoder_input')
flattened = Flatten(name='encoder_flatten')(input_img)
encoded1 = Dense(128, activation='relu', name='encoder_dense1')(flattened)
encoded2 = Dense(64, activation='relu', name='encoder_dense2')(encoded1)
bottleneck = Dense(latent_dim, activation='relu', name='bottleneck')(encoded2)

encoder = Model(input_img, bottleneck, name='encoder')
encoder.summary()

In [ ]:
# Decoder
decoder_input = Input(shape=(latent_dim,), name='decoder_input')
decoded1 = Dense(64, activation='relu', name='decoder_dense1')(decoder_input)
decoded2 = Dense(128, activation='relu', name='decoder_dense2')(decoded1)
decoded3 = Dense(784, activation='sigmoid', name='decoder_output')(decoded2)
reconstructed = Reshape((28, 28), name='decoder_reshape')(decoded3)

decoder = Model(decoder_input, reconstructed, name='decoder')
decoder.summary()

In [ ]:
# Autoencoder (combining encoder and decoder)
autoencoder_input = Input(shape=(28, 28), name='autoencoder_input')
encoded_repr = encoder(autoencoder_input)
reconstructed_img = decoder(encoded_repr)
autoencoder = Model(autoencoder_input, reconstructed_img, name='autoencoder')
autoencoder.summary()

In [ ]:
from tensorflow.keras import losses
autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())

In [ ]:
autoencoder.fit(x_train, x_train,
                epochs=10,
                batch_size=256,
                shuffle=True,
                validation_data=(x_test, x_test))

In [ ]:
# Visualize original and reconstructed Fashion MNIST images
decoded_imgs = autoencoder.predict(x_test)

In [ ]:
n = 10  # Number of images to display
plt.figure(figsize=(20, 4))
for i in range(n):
    # Display original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Original', fontsize=14)
    # Display reconstruction
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(decoded_imgs[i].reshape(28, 28), cmap='gray')
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    if i == 0: ax.set_ylabel('Reconstructed', fontsize=14)
plt.show()

# Variational Autoencoder (VAE)

 A VAE is a probabilistic take on the autoencoder, a model which takes high dimensional input data and compresses it into a smaller representation. Unlike a traditional autoencoder, which maps the input onto a latent vector, a VAE maps the input data into the parameters of a probability distribution, such as the mean and variance of a Gaussian. This approach produces a continuous, structured latent space, which is useful for image generation.

## Define the Sampling Layer
First, we need a way to sample from the distribution our encoder creates. The "reparameterization trick" is used here, which is essential for allowing the gradients to flow back through the network during training. This can be implemented as a custom Keras layer.

In [ ]:
# VAE: Define the Sampling Layer
class Sampling(tf.keras.layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

## Rebuild the Encoder for the VAE

The VAE encoder is slightly different. Instead of one output (the bottleneck), it will have two: `z_mean` and `z_log_var` (the log of the variance). These define our latent space distribution.

### VAE Encoder and Sampling

Unlike a standard Autoencoder which maps an input image to a **fixed vector** in the latent space, a Variational Autoencoder (VAE) maps an input to a **probability distribution**.

For each input image, the encoder outputs two vectors:
1.  `z_mean`: The center (mean) of the distribution.
2.  `z_log_var`: The spread (log variance) of the distribution.

**The Reparameterization Trick:**
We want to sample a point `z` from this distribution to feed into the decoder. However, we cannot backpropagate gradients through a random sampling operation. 

**Solution:** We move the randomness to a separate variable `epsilon` sampled from a standard normal distribution. Then we compute:
> `z = z_mean + epsilon * sigma`

This allows the network to learn `z_mean` and `z_log_var` via standard backpropagation.

In [ ]:
# VAE: Define the Encoder
latent_dim = 2  # Using a 2D latent space to easily visualize it

# Original input
encoder_inputs = Input(shape=(28, 28), name='vae_encoder_input')
x = Flatten()(encoder_inputs)
x = Dense(128, activation='relu')(x)

# VAE specific outputs
z_mean = Dense(latent_dim, name='z_mean')(x)
z_log_var = Dense(latent_dim, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])

# Instantiate the encoder model
vae_encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name='vae_encoder')
vae_encoder.summary()

## Reuse the Decoder
The great part is that you can reuse the exact same decoder architecture you defined earlier! The only change is that we'll create a new instance of it that connects to the output of our new VAE encoder.

In [ ]:
# VAE: Define the Decoder (reusing the same architecture)
latent_inputs = Input(shape=(latent_dim,), name='vae_decoder_input')
x = Dense(64, activation='relu')(latent_inputs)
x = Dense(128, activation='relu')(x)
x = Dense(784, activation='sigmoid')(x)
outputs = Reshape((28, 28))(x)

# Instantiate the decoder model
vae_decoder = Model(latent_inputs, outputs, name='vae_decoder')
vae_decoder.summary()

## Define the VAE as a Custom Model with Loss

The VAE loss function is unique because it balances two competing goals:

1.  **Reconstruction Loss (Accuracy):**
    *   Measures how well the decoded image matches the original input (pixel-by-pixel).
    *   We use `binary_crossentropy` for this.
    *   *Effect:* Makes the generated digits look like real digits.

2.  **KL Divergence Loss (Regularization):**
    *   Measures how much the learned latent distribution deviates from a standard normal distribution (Gaussian with mean 0, variance 1).
    *   *Effect:* Forces the latent space to be **continuous** and **smooth**. This ensures that if we sample from gaps between known digits, we still get a valid-looking digit (morphing), rather than noise.

We combine these using a weight `kl_weight` to control the trade-off.

In [ ]:
# VAE: Define the full VAE model with a weighted custom loss
class VAE(tf.keras.Model):
    # Add a kl_weight argument to the constructor
    def __init__(self, encoder, decoder, kl_weight=1.0, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.kl_weight = kl_weight  # Store the weight
        self.total_loss_tracker = tf.keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = tf.keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = tf.keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    tf.keras.losses.binary_crossentropy(data, reconstruction), axis=(1)
                )
            )
            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            
            # *** THE KEY CHANGE IS HERE ***
            # Apply the weight to the KL loss before adding it
            total_loss = reconstruction_loss + self.kl_weight * kl_loss
            
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

The `kl_loss` in a Variational Autoencoder (VAE) is the Kullback-Leibler (KL) divergence between the learned latent distribution and a standard normal distribution.

__Purpose__:

* It regularizes the encoder so that the latent space is continuous and well-structured.
* It encourages the encoded vectors (mean and variance) to be close to a normal distribution (mean 0, variance 1).

__Why is this important?__

* Without KL loss, the encoder could map each input to a separate point, making the latent space discontinuous.
* With KL loss, you can sample new points from the latent space and generate realistic outputs.

> `kl_loss` keeps the latent space organized and enables generative capabilities in VAEs.

In [ ]:
# Instantiate the VAE with a KL weight
# This is a hyperparameter you can tune. Start small.
vae = VAE(vae_encoder, vae_decoder, kl_weight=0.01)
vae.compile(optimizer=tf.keras.optimizers.Adam())


In [ ]:
vae.summary()

## Train the VAE and Visualize the Latent Space

In [ ]:
# Load MNIST data again (if needed, ensure it's flattened for VAE)
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

# Train the VAE
# 20-30 epochs is usually good for a simple VAE on MNIST
vae.fit(x_train, epochs=20, batch_size=128)

## 4. Visualize Latent Space Distribution

Let's visualize how the encoder maps digits into the 2D latent space. We'll encode a batch of test images and plot their mean latent vectors (`z_mean`), coloring the points by their digit class. This will show us how well the VAE clusters similar digits.

In [ ]:
# Visualizing the latent space with clusters
def plot_label_clusters(vae, data, labels):
    # display a 2D plot of the digit classes in the latent space
    z_mean, _, _ = vae.encoder.predict(data, verbose=0)
    plt.figure(figsize=(12, 10))
    plt.scatter(z_mean[:, 0], z_mean[:, 1], c=labels)
    plt.colorbar()
    plt.xlabel("z[0]")
    plt.ylabel("z[1]")
    plt.show()

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_test = x_test.astype("float32") / 255  # Shape stays (n, 28, 28) — matches VAE encoder input

plot_label_clusters(vae, x_test, y_test)

## 5. Generate New Images (Latent Space Manifold)

Now we'll test the generative capabilities of the VAE. We will sample points from the latent space in a grid pattern and decode them. This allows us to visualize the continuous manifold of digits the VAE has learned.

In [ ]:
# Visualize how the digits are clustered in the latent space
def plot_latent_space(vae, n=30, figsize=15):
    digit_size = 28
    scale = 1.0
    figure = np.zeros((digit_size * n, digit_size * n))
    
    # Create a grid of 2D points
    grid_x = np.linspace(-scale, scale, n)
    grid_y = np.linspace(-scale, scale, n)[::-1]

    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = np.array([[xi, yi]])
            x_decoded = vae.decoder.predict(z_sample, verbose=0)
            digit = x_decoded[0].reshape(digit_size, digit_size)
            figure[
                i * digit_size : (i + 1) * digit_size,
                j * digit_size : (j + 1) * digit_size,
            ] = digit

    plt.figure(figsize=(figsize, figsize))
    plt.imshow(figure, cmap="Greys_r")
    plt.axis("Off")
    plt.show()

plot_latent_space(vae)

This grid is a map of your VAE's "understanding" of the MNIST digits. Here's a breakdown of what you're seeing:

* __Clustering of Similar Digits__: You can clearly see that the model has grouped similar digits together. There are distinct regions for '9's, '8's, '5's, '3's, '6's, and '0's. This is a sign of a well-organized latent space.

* __Smooth Transitions__: Notice how the digits gradually morph into one another as you move across the grid. For instance, the '8's on the top right seem to blend into the '1's. Similarly, the '3's slowly transform into '2's. This continuous transition is a hallmark of a successful VAE and demonstrates its generative capabilities.

* __Generative Power__: Each image in this grid is a brand-new, computer-generated digit. Your model can now generate a vast array of unique digits by simply picking a coordinate in this latent space and feeding it to the decoder.